# What a summary throws away

**Scenario:** a chip fab runs an assistant over the shift log. Overnight it summarises the log for the
morning shift. On Tuesday the summary said a lot was on hold and did not say which one. The follow up
question got an answer anyway, and the answer named the wafer as the lot.

Compaction is shrinking a long history so it still fits. Think of it as a set of meeting minutes.
Minutes keep the decisions and drop the phone numbers, which is fine until somebody has to call.

## Mechanics

Four ways to make a long history fit, and they fail in different places.

| Strategy | What it keeps | What it loses | Cost per compaction |
|---|---|---|---|
| Sliding window | the last few turns, exactly | everything older, all at once | nothing |
| Progressive summary | a rewritten gist | exact strings, and detail from the middle | one model call |
| Persistent facts | values you named and pinned | anything you did not name | nothing, but you must choose |
| External scratchpad | a file the run reads and writes | nothing, until nobody reads it | one tool call |

Only the window and the scratchpad keep a string byte for byte. A summary is a rewrite, and a rewrite
may drop a serial number it judged unimportant.

## The picture

![A summary keeps the gist and drops the exact strings](images/summary-loss.svg)

The pinned facts sit outside the part that gets rewritten. That is the whole idea, and the code below
is one way to build it.

## The cost

```
survival = summaries where the exact id came back / summaries tried
tokens   = pinned facts + summary of the old turns + the last few turns, verbatim
```

Both come from real responses below. Survival is the number that matters, because a history that fits
and lies is worse than one that did not fit.

## The failure

A night shift log. Forty routine lines, and one line that a person would never lose.

In [1]:
LOT, WAFER = "LOT-7F2A-1183", "W-24-0917"
NEEDLE = (f"Killer defect cluster confirmed on wafer {WAFER} from {LOT}. "
          "Root cause is a cracked focus ring in etcher 7. Hold the whole lot.")
ROUTINE = [
    "Chamber {n} pressure drifted slightly overnight, still inside tolerance.",
    "Metrology queue is {n} minutes behind, nothing blocking.",
    "Photoresist batch swapped on track {n}, no recipe change.",
    "Brief vacuum alarm on implanter {n}, cleared itself.",
    "CMP pad replaced on polisher {n}, first article looks clean.",
    "Defect density on run {n} sits near the line average.",
    "Etch uniformity map for tool {n} shows the usual edge roll off.",
    "Litho overlay on stepper {n} within spec, no rework requested.",
    "Furnace {n} ramp profile unchanged since the last qual.",
    "Wet bench {n} chemistry refreshed on schedule.",
    "Handler jam on sorter {n}, two minutes, no wafers lost.",
    "Probe card {n} cleaned, contact resistance back to normal.",
]


def shift_log(position, count=40):
    """Forty routine lines with the one line that matters placed where you say."""
    turns = [ROUTINE[i % len(ROUTINE)].format(n=i + 1) for i in range(count)]
    turns.insert({"start": 1, "middle": count // 2, "end": count - 1}[position], NEEDLE)
    return [f"[{i:03d}] {t}" for i, t in enumerate(turns)]

The summariser is a small cheap model, the normal choice. Compaction runs on every long conversation,
so nobody pays a large model for it.

In [2]:
from vault import get_client, load_env, model_for

load_env()
client = get_client("04-context-engineering/02-what-a-summary-throws-away")


def summarise(turns):
    """Rewrite a long log into a short one. This is where strings go to die."""
    reply = client.chat.completions.create(
        model=model_for("small"), max_tokens=220,
        messages=[{"role": "system", "content": "Summarise this fab shift log in under 100 words."},
                  {"role": "user", "content": "\n".join(turns)}])
    return reply.choices[0].message.content

Run it with the important line in three different places. Same log, same summariser. The only thing
that changes is where the line sits.

In [3]:
survival = {}
for position in ("start", "middle", "end"):
    summaries = [summarise(shift_log(position)) for _ in range(4)]
    survival[position] = summaries
    kept = sum(LOT in text for text in summaries)
    print(f"needle at the {position:6}: the exact lot id survived {kept} of {len(summaries)}")

needle at the start : the exact lot id survived 1 of 4
needle at the middle: the exact lot id survived 0 of 4
needle at the end   : the exact lot id survived 2 of 4


Now the part that reaches a person. The next question is asked against the summary, not the log, so
anything the summary dropped is gone.

In [4]:
written = [text for group in survival.values() for text in group]
kept = sum(LOT in text for text in written)
same_digits = sum(LOT not in t and LOT.lower() in t.lower() for t in written)
lost = min(written, key=lambda t: (LOT.lower() in t.lower(), LOT in t))
question = "Which lot is on hold and why? Give the lot id exactly."

reply = client.chat.completions.create(
    model=model_for("small"), max_tokens=120,
    messages=[{"role": "system", "content": "Answer only from the notes given."},
              {"role": "user", "content": f"{lost}\n\n{question}"}])

print(f"of {len(written)} summaries: {kept} kept it exactly, {same_digits} changed the case, "
      f"{len(written) - kept - same_digits} lost it")
print("\ndownstream answer:", reply.choices[0].message.content.strip())
assert kept == len(written), f"the lot id was lost in {len(written) - kept} of {len(written)} summaries"

of 12 summaries: 3 kept it exactly, 0 changed the case, 9 lost it

downstream answer: Lot W-24-0917 is on hold due to a killer defect cluster caused by a cracked focus ring in etcher 7.


AssertionError: the lot id was lost in 9 of 12 summaries

## The diagnosis

The assertion fires. Nine of the twelve summaries came back with no lot id at all. The second counter
is there for the other way this fails, which earlier runs did produce: the digits survive and the
case changes, and a tracking system reads that as a different string.

The downstream step never notices either way. Asked which lot is on hold, it answers from notes with
no lot id in them and offers the wafer id instead. Somebody holds the wrong material on that.

Look at the mechanics table again. A rewrite ranks. Routine lines that repeat forty times read as the
shape of the shift, so they survive as a theme. One serial number that appears once reads as noise.

Position is the second effect, and it is weaker than the folklore. Repeat the sweep and the counts
move. One run held the front every time and the middle almost never. Another lost the front as often
as anywhere else. Treat position as a tendency you cannot lean on.

## The fix

Do not ask the summariser to be careful. Take the exact strings out of the history before it is
rewritten, and hold them somewhere the rewrite cannot reach.

In [5]:
import re

ID_PATTERN = re.compile(r"\b(?:LOT|W|RCP)-[A-Z0-9]+(?:-[A-Z0-9]+)*\b")


def pin_identifiers(turns):
    """Lift every exact id out of the history, with the line it came from."""
    pinned = {}
    for turn in turns:
        for found in ID_PATTERN.findall(turn):
            pinned.setdefault(found, turn.split("] ", 1)[-1])
    return pinned

A regular expression is enough here because the ids have a shape. Where they do not, the tool that
wrote the turn knows what it wrote, and pinning belongs there instead.

Then the compaction itself, which is three blocks in a deliberate order.

In [6]:
def compact(turns, summarise_fn, keep_last=3):
    """Pinned facts, then a summary of the old turns, then the last turns verbatim."""
    older, recent = turns[:-keep_last], turns[-keep_last:]
    facts = "\n".join(f"{name}: {line}" for name, line in pin_identifiers(turns).items())
    return ("## Pinned facts, never summarised\n" + facts
            + "\n\n## Summary of earlier turns\n" + summarise_fn(older)
            + "\n\n## Last turns, word for word\n" + "\n".join(recent))

The summariser is an argument now, not a hard wired call. That is what lets the gate below hand it
one that destroys everything and still expect the ids through.

Same question, against the compacted history instead of the raw summary.

In [7]:
packed = compact(shift_log("middle"), summarise)
answers = []
for _ in range(4):
    reply = client.chat.completions.create(
        model=model_for("small"), max_tokens=120,
        messages=[{"role": "system", "content": "Answer only from the notes given."},
                  {"role": "user", "content": f"{packed}\n\n{question}"}])
    answers.append((reply.choices[0].message.content, reply.usage.prompt_tokens))

exact = sum(LOT in text for text, _ in answers)
print(f"before: the exact lot id survived {kept} of {len(written)} summaries")
print(f"after : the exact lot id is in {exact} of {len(answers)} answers")
print(f"        {answers[0][1]} prompt tokens for the compacted history")
print(f"sample: {answers[0][0].strip()[:150]}")

before: the exact lot id survived 3 of 12 summaries
after : the exact lot id is in 4 of 4 answers
        442 prompt tokens for the compacted history
sample: LOT-7F2A-1183 is on hold due to a killer defect cluster confirmed on wafer W-24-0917 from the same lot. The root cause is a cracked focus ring in etch


## The gate

The check has to hold whatever the summariser does. So the gate hands it the worst one possible and
still expects the ids.

In [8]:
def test_pinned_ids_survive_a_destructive_summary():
    turns = [f"[000] {NEEDLE}"] + [f"[{i:03d}] routine note {i}" for i in range(1, 8)]
    packed = compact(turns, lambda older: "nothing of note happened earlier")
    assert LOT in packed, "the lot id did not survive compaction"
    assert WAFER in packed, "the wafer id did not survive compaction"


test_pinned_ids_survive_a_destructive_summary()
print("gate holds: a summariser that deletes everything cannot delete a pinned id")

gate holds: a summariser that deletes everything cannot delete a pinned id


Move the pinned block inside the text handed to `summarise_fn` and this test fails on the first
assertion.

### Enterprise exploration

- Pinned facts grow forever while the history is capped. What evicts a fact, and who signs off that
  an id is no longer needed?
- A pattern finds ids that match a shape. What does the one that does not match cost, and how would
  you find out it was missed?
- Every compaction is a model call on the critical path. What does that add to latency at your
  busiest hour, and what happens when the summariser is down?
- A hold decision traced back to a summary, not the log. What does an auditor need to replay that
  decision, and where is it kept?

### Key takeaways

- A summary is a rewrite. It keeps themes and drops values that appear once.
- Where a fact sits nudges the odds, not enough to rely on.
- A model asked about a missing id answers anyway, from whatever is nearest.
- Pin exact strings outside the rewrite, and prove it with a summariser that deletes.